In [1]:
from keras.layers import Dense, Dropout
from keras.models import Sequential
from keras.optimizers import Adam
import pandas as pd 
import tensorflow as tf
from tensorflow.keras.regularizers import l2
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,accuracy_score
from keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder



In [4]:
df = pd.read_csv('Thien_Uu.csv')
df.head()

,GIST_0,GIST_1,GIST_2,GIST_3,GIST_4,GIST_5,GIST_6,GIST_7,GIST_8,GIST_9,...,GIST_119,GIST_120,GIST_121,GIST_122,GIST_123,GIST_124,GIST_125,GIST_126,GIST_127,class
0,0.001387,0.000820,0.001319,0.000786,0.001090,0.001229,0.001423,0.001787,0.001208,0.017221,...,0.001520,0.001119,0.001531,0.006161,0.002631,0.002506,0.033151,0.024942,0.002070,positive
1,0.014555,0.028314,0.026516,0.004162,0.001652,0.001647,0.004622,0.012038,0.002175,0.034284,...,0.006209,0.000769,0.001208,0.001088,0.000918,0.001917,0.011253,0.050512,0.004258,positive
2,0.002258,0.009734,0.001074,0.001196,0.001346,0.000902,0.001010,0.001091,0.003414,0.046807,...,0.001990,0.000785,0.003204,0.008465,0.007019,0.000827,0.013845,0.047319,0.001672,positive
3,0.000959,0.001216,0.002057,0.001346,0.001068,0.001529,0.001984,0.001013,0.001332,0.002051,...,0.002557,0.001676,0.000775,0.000469,0.000762,0.002117,0.028958,0.031339,0.001867,positive
4,0.001517,0.000951,0.000973,0.001367,0.001424,0.000976,0.001300,0.001189,0.001418,0.002512,...,0.003148,0.001907,0.003551,0.002413,0.003461,0.002297,0.022082,0.041486,0.001978,positive


In [5]:
X = df.drop(columns=['class'])
y = df['class']

le = LabelEncoder()
y = le.fit_transform(y)


In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.fit_transform(X_val)
X_test_scaled = scaler.fit_transform(X_test)

In [7]:
model = Sequential()
model.add(Dense(units=128, activation='relu', input_shape=(X_train.shape[1],),kernel_regularizer=l2(0.01)))
model.add(Dense(units=64, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dense(units=32, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dense(units=1, activation='sigmoid', kernel_regularizer=l2(0.01)))

model.compile(optimizer = Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               16512     
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 1)                 33        
                                                                 
Total params: 26,881
Trainable params: 26,881
Non-trainable params: 0
_________________________________________________________________


In [8]:
call_back = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

model.fit(X_train_scaled, 
          y_train, 
          epochs=200, 
          batch_size=16, 
          validation_data=(X_val_scaled, y_val),
          callbacks = [call_back],
          verbose = 1)

Epoch 1/200
88/88 [==============================] - 2s 6ms/step - loss: 1.0289 - accuracy: 0.8575 - val_loss: 0.4571 - val_accuracy: 0.9103
Epoch 2/200
88/88 [==============================] - 0s 4ms/step - loss: 0.4843 - accuracy: 0.8846 - val_loss: 0.4201 - val_accuracy: 0.9369
Epoch 3/200
88/88 [==============================] - 0s 4ms/step - loss: 0.4448 - accuracy: 0.8917 - val_loss: 0.4096 - val_accuracy: 0.9070
Epoch 4/200
88/88 [==============================] - 0s 4ms/step - loss: 0.3918 - accuracy: 0.9145 - val_loss: 0.4259 - val_accuracy: 0.8937
Epoch 5/200
88/88 [==============================] - 0s 4ms/step - loss: 0.3951 - accuracy: 0.9024 - val_loss: 0.3569 - val_accuracy: 0.9236
Epoch 6/200
88/88 [==============================] - 0s 4ms/step - loss: 0.3927 - accuracy: 0.9067 - val_loss: 0.3371 - val_accuracy: 0.9402
Epoch 7/200
88/88 [==============================] - 0s 5ms/step - loss: 0.3822 - accuracy: 0.9095 - val_loss: 0.3458 - val_accuracy: 0.9369
Epoch 8/200
8

In [34]:
train_loss, train_acc = model.evaluate(X_train_scaled, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val_scaled, y_val, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test, verbose=0)

print(f"Acc on Training set: {train_acc * 100:2f}%")
print(f"Acc on validation set: {val_acc * 100:2f}%")
print(f"Acc on Test set: {test_acc * 100:2f}%")


Acc on Training set: 94.230771%
Acc on validation set: 93.687707%
Acc on Test set: 93.023258%


Với việc thay đổi kiến trúc của mạng ANN và việc sử dụng và thay đổi các giá trị của hyper parameter (learning rate, epochs và regularization,.....).

Dựa vào kết quả trên ta thấy được sự khác biệt giữa accuracy của traing set và testing set không quá lớn khoản 1-2%. điều này cho thấy model khoogn có sự overfitting không rõ ràng. Việc thay đổi kiến trúc và sử dụng hợp lí các hyper parameter giúp cho acc của traiing khá cao (94.23) và acc của testing (93.02%). So sánh acc trên 3 tập đều tưởng đối cao và không chênh lệch nên model không cho thấy dấu hiệu của under fitting (hightbias)